# 01 - 最小可运行的 CrewAI

本 notebook 演示 CrewAI 的最小工作单元:

- 1 个 **Tool**(可调用的 Python 函数)
- 1 个 **Agent**(LLM + tools + 角色定义)
- 1 个 **Task**(给 agent 的指令)
- 1 个 **Crew**(把 agent 和 task 装到一起)
- `crew.kickoff()` 跑一次

跑完这个 notebook,你就能理解生产代码中 8 代理 `AnalysisCrew`
的每一层在做什么。

**前置条件**:`.env` 中设置了 `MINIMAX_API_KEY` 和 `LITELLM_MODEL`。
本 notebook 每次 LLM 调用大约 2-5 秒,总共 1-3 次调用。


In [1]:
from
alphaquant.infrastructure.config import get_settings
from alphaquant.infrastructure.llm import get_llm

settings = get_settings()
print(f"Model: {settings.litellm_model}")
llm = get_llm(temperature=0.1)
print("LLM ready")


Model: openai/MiniMax-M2.7-highspeed
LLM ready


In [2]:
from crewai.tools import tool

@tool("echo")
def echo_tool(text: str) -> str:
    """回显输入文本。用于演示 tool 调用。"""
    return f"echo: {text}"

# tool 本身就是一个可调用的对象,可以直接 run
print(echo_tool.run(text="hello"))


Using Tool: echo
echo: hello


In [3]:
from crewai import Agent

agent = Agent(
    role="回声员",
    goal="把用户输入回显出来",
    backstory="你是一个简单的回声员,负责把用户的输入原样返回。",
    tools=[echo_tool],
    llm=llm,
    verbose=False,  # 关闭 verbose,这里用 notebook 自己的 cell 观察
)
print(f"Agent: {agent.role}")


Agent: 回声员


In [4]:
from crewai import Task

task = Task(
    description="用 echo 工具回显文本 'Hello, CrewAI!'",
    expected_output="回显后的文本",
    agent=agent,
)
print(f"Task assigned to: {task.agent.role}")


Task assigned to: 回声员


In [5]:
from crewai import Crew, Process

crew = Crew(
    agents=[agent],
    tasks=[task],
    process=Process.sequential,
    verbose=True,  # 这里打开 verbose 看到内部步骤
)
result = crew.kickoff(inputs={})
print("---")
print("Result type:", type(result).__name__)
print("Result.raw:")
print(result.raw)


╭──────────────────────────────────────────── Crew Execution Started ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: a7235d6e-c9bf-442b-a7e3-e001a00efca4                                                                       │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: 回声员                                                                                                  │
│                                                                                                                 │
│  Task: 用 echo 工具回显文本 'Hello, CrewAI!'                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Output()

╭──────────────────────────────────────────── 🔧 Agent Tool Execution ────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: 回声员                                                                                                  │
│                                                                                                                 │
│  Thought: Thought: I need to use the echo tool to return the text 'Hello, CrewAI!' as requested.                │
│                                                                                                                 │
│  Using Tool: echo                                                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────────── Tool Input ───────────────────────────────────────────────────╮
│                                                                                                                 │
│  {                                                                                                              │
│    "text": "Hello, CrewAI!"                                                                                     │
│  }                                                                                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: a7235d6e-c9bf-442b-a7e3-e001a00efca4                                                                       │
│  Tool Args:                                                                                                     │
│  Final Output: Hello, CrewAI!                                                                                   │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── Execution Traces ────────────────────────────────────────────────╮
│                                                                                                                 │
│  🔍 Detailed execution traces are available!                                                                    │
│                                                                                                                 │
│  View insights including:                                                                                       │
│    • Agent decision-making process                                                                              │
│    • Task execution flow and timing                                                                             │
│    • Tool usage details                                                                                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Would you like to view your execution traces? [y/N] (20s timeout): 

Exception in thread Thread-10 (get_input):
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1073, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1010, in run
    self._target(*self._args, **self._kwargs)
  File "/home/lisong/projects/investment-agent/.venv/lib/python3.12/site-packages/crewai/events/listeners/tracing/utils.py", line 359, in get_input
    response = input().strip().lower()
               ^^^^^^^
  File "/home/lisong/projects/investment-agent/.venv/lib/python3.12/site-packages/ipykernel/kernelbase.py", line 1402, in raw_input
    raise StdinNotImplementedError(msg)
IPython.core.error.StdinNotImplementedError: raw_input was called, but this frontend does not support input requests.


---
Result type: CrewOutput
Result.raw:
Hello, CrewAI!


## 你刚才看到了什么

1. `crew.kickoff(inputs={})` 启动 crew
2. 因为 `process=sequential`,只有一个 task,manager 没有委托,agent 直接拿到 task
3. Agent 决定调用 echo 工具,传 `'Hello, CrewAI!'`
4. Tool 返回 `'echo: Hello, CrewAI!'`
5. Agent 把工具结果作为最终输出
6. `CrewOutput` 对象包含 `.raw`(原始文本)、`.tasks_output`(每个 task 的输出)等

## 跟生产代码的对应关系

| 教学示例 | AnalysisCrew 中的位置 |
|---|---|
| `echo_tool` | `tools/company_lookup_tool.py` 等 5 个真实工具 |
| 回声员 | `agents/company_resolver.py` 等 8 个真实 agent |
| 1 个 task | `_TASK_TEMPLATES` 中 8 个真实 task |
| `Crew(...)` | `AnalysisCrew.crew = self._build_crew()` |
| `kickoff()` | `AnalysisCrew.kickoff()` → `Flow.run()` |
